# Demo: Unterrichtsblock 4

Kurze Transferdemo für fehlende Werte, Duplikate und Zeitindex.

Du arbeitest hier mit synthetischen Daten von zwei Lüftungsanlagen (RLT) in einem Schulgebäude. Anschliessend bearbeitest du selbstständig `01-duplikate-fehlende_werte.ipynb` und `02-timeindex.ipynb`.

Relevante Artikel in der Wissensdatenbank:

- [Wissensdatenbank: Entfernen von Duplikaten](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/drop_duplicates/)
- [Wissensdatenbank: Befüllen von fehlenden Werten](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/fillna/)
- [Wissensdatenbank: Time-Indexes](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/time_indexes/)

In [ ]:
import pandas as pd

raw = pd.read_csv("../../data/demo_rlt_school_building_raw.csv")
raw.head()

### Aufgabe 1
Bereinige fehlende Werte und Duplikate. Begründe für jede Spalte, welcher Füllwert fachlich sinnvoll ist.

- Kontext: zwei RLT-Anlagen im Schulhaus
- Überblick: wie viele Werte fehlen, Wertebereich pro Spalte
- fehlende Werte bei `MANUAL_OVERRIDE_MIN`, `ALARM_COUNT` und `EXTRACT_CO2_PPM`
- kein Eintrag bei Eingriff oder Alarm heisst: nichts passiert
- CO2-Sensorausfall: 0 ppm ist unmöglich
- Duplikate über `MEASURED_AT_LOCAL` + `AHU_ID`

In [ ]:
# Dein Code hier

In [ ]:
raw.isna().sum()

In [ ]:
# count zeigt die Lücken pro Spalte, min/max zeigen unplausible Werte
raw.describe()

In [ ]:
df_clean = raw.copy()
# kein Eintrag = kein manueller Eingriff, keine Alarmmeldung
df_clean["ALARM_COUNT"] = df_clean["ALARM_COUNT"].fillna(0)
df_clean["MANUAL_OVERRIDE_MIN"] = df_clean["MANUAL_OVERRIDE_MIN"].fillna(0)

In [ ]:
# Sensorausfall: 0 ppm wäre falsch, der Mittelwert ist eine einfache Schätzung
co2_mean = df_clean["EXTRACT_CO2_PPM"].mean()
co2_mean

In [ ]:
df_clean["EXTRACT_CO2_PPM"] = df_clean["EXTRACT_CO2_PPM"].fillna(co2_mean)
df_clean

In [ ]:
# ganze Zeile identisch: findet weniger Duplikate als über die Schlüsselspalten
df_clean.duplicated().sum()

In [ ]:
df_clean.duplicated(subset=["MEASURED_AT_LOCAL", "AHU_ID"]).sum()

In [ ]:
df_clean[df_clean.duplicated(subset=["MEASURED_AT_LOCAL", "AHU_ID"])]

In [ ]:
df_clean = df_clean.drop_duplicates(subset=["MEASURED_AT_LOCAL", "AHU_ID"], keep="first")
print("Zeilen vorher:", len(raw), "| Zeilen nachher:", len(df_clean))

df_clean

### Aufgabe 2
Baue aus den bereinigten Daten eine Zeitreihe und zeige nur die Messungen zwischen 07:00 und 07:30.

- `MEASURED_AT_LOCAL` ist zuerst nur Text
- echte Zeitinformation
- als Index setzen und sortieren
- zeitbasierte Slices

In [ ]:
# Dein Code hier

In [ ]:
df_time = df_clean.copy()

df_time["MEASURED_AT_LOCAL"].dtype

In [ ]:
df_time["MEASURED_AT_LOCAL"]

In [ ]:
df_time["MEASURED_AT_LOCAL"] = pd.to_datetime(df_time["MEASURED_AT_LOCAL"], format="%d.%m.%Y %H:%M")

df_time

In [ ]:
df_time = df_time.set_index("MEASURED_AT_LOCAL")
df_time.index

In [ ]:
df_time = df_time.sort_index()

df_time.head()

In [ ]:
df_time["2026-01-15 07:00":"2026-01-15 07:30"]

### Aufgabe 3
Interpretiere die Zeitstempel als Schweizer Ortszeit und rechne sie in UTC um. Was passiert bei der Umstellung auf Sommerzeit?

- naive Zeit kennt keine Zeitzone
- `Europe/Zurich` zuweisen
- in UTC umrechnen
- Frühling: 02:00 bis 02:59 fehlt

In [ ]:
# Dein Code hier

In [ ]:
df_time.index

In [ ]:
df_local = df_time.copy()
df_local = df_local.tz_localize("Europe/Zurich")

df_local.index

In [ ]:
df_utc = df_local.tz_convert("UTC")

df_utc.index

In [ ]:
# Messungen während der Umstellung auf Sommerzeit
dst_spring = pd.read_csv("../../data/ub4_demo_rlt_zeitumstellung.csv")
dst_spring["MEASURED_AT_LOCAL"] = pd.to_datetime(dst_spring["MEASURED_AT_LOCAL"], format="%d.%m.%Y %H:%M")
dst_spring = dst_spring.set_index("MEASURED_AT_LOCAL")
dst_spring

In [ ]:
# ohne nonexistent= gibt es einen NonExistentTimeError: 02:00 bis 02:59 gibt es am 29.03. nicht
dst_spring = dst_spring.tz_localize("Europe/Zurich", nonexistent="NaT")

dst_spring

In [ ]:
dst_spring.tz_convert("UTC")
